In [1]:
#imports
#python
import os, sys
import scipy
from datetime import datetime
import re
from collections import defaultdict, OrderedDict
import itertools
# image and data processing
import torch
from torchvision.models import alexnet
from torchvision import transforms as T
from PIL import Image , ImageOps
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import scipy.optimize as optimize
from scipy.linalg import block_diag
from scipy.spatial.distance import cdist
from munkres import Munkres, print_matrix
#qiskit and d-wave
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo



In [2]:
#search for and find the files
def path_joiner(image_name, root_dir = None): #recursive search for the image
    if root_dir is None:
        root_dir = os.getcwd()
    for dirpath , _, filenames in os.walk(root_dir):
        if image_name in filenames:
            return os.path.join(dirpath, image_name)
#usage
mat_files = ['060_0000.mat', '060_0010.mat', '060_0013.mat']
paths = []
for fi in mat_files:
    paths.append(path_joiner(fi))

num_views = len(mat_files)
num_keypoints = 4
print(num_views)

3


In [3]:
#Keypoints extraction
def keypoint(image_path, max_points=None): # extract a desired number of keypoints from the images
    keypoints1 = scipy.io.loadmat(image_path)["pts_coord"]
    if max_points is not None:
        keypoints1 = keypoints1[:, :max_points]
    return keypoints1
def keypoints_dict(image_names: list, max_points: int):
    keypoints = {}
    for image_name in image_names:
        base_name = os.path.splitext(image_name)[0]
        full_path = path_joiner(image_name)
        if full_path:
            kps = keypoint(full_path, max_points)
            keypoints[base_name] = kps
        else:
            print(f"[Warning] image not found: {image_name}")
    return keypoints
points_dic = keypoints_dict(mat_files,max_points=num_keypoints)
print(f'This is the keypoints dict {points_dic}')

This is the keypoints dict {'060_0000': array([[276.29990715, 240.07950325, 214.77483751, 212.29398793],
       [ 41.45485144,  25.57741411,  15.15784587,  92.56035283]]), '060_0010': array([[232.21170177, 202.86387568, 190.90735394, 177.86387568],
       [ 62.04144022,  37.40375906,  26.53419384,  66.75158514]]), '060_0013': array([[228.41360294, 193.07536765, 173.44301471, 163.13602941],
       [ 68.35202206,  48.71966912,  44.30238971,  93.87408088]])}


In [4]:
#Feature extraction using AlexNet

def alexnet_feature_extractor(layer= 'conv4'):
    model = alexnet(pretrained=True)
    model.eval()
    if layer == 'conv4':
        return torch.nn.Sequential(*list(model.features)[:10])
    elif layer == 'conv5':
        return torch.nn.Sequential(*list(model.features)[:12])
    else:
        raise ValueError("Invalid layer. Choose 'conv4' or 'conv5'.")
def extract_features(keypoints_dict, patch_size=64, layer='conv4'):
    feature_extractor = alexnet_feature_extractor(layer)
    transform = T.Compose([
        T.Resize((227, 227)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    feature_extractor.to(device)
    features = {}

    for image_name, keypoints in keypoints_dict.items():
        img_path = path_joiner(image_name + '.png')
        img = Image.open(img_path).convert('RGB')
        # img = Image.open(img_path).convert('L')
        img_np = np.array(img)
        feature_list = []

        for (x, y) in keypoints.T:
            half = patch_size // 2

            # Pad the image if keypoint is near the edge
            padding = (half, half, half, half)  # left, top, right, bottom
            padded_img = ImageOps.expand(img, border=padding, fill=0)

            # shift keypoint for padding the edges
            x_padded = x + half
            y_padded = y + half

            # safely crop the full patch_size
            patch = padded_img.crop((x_padded - half, y_padded - half,
                                    x_padded + half, y_padded + half))
            # patch = patch.convert('RGB')  # duplicate L channel into R, G, B for alexnet
            patch_tensor = transform(patch).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = feature_extractor(patch_tensor)
                feat = feat.mean(dim=[2, 3]) # to flatten the output dimensions
            feature_list.append(feat.squeeze().cpu().numpy())

        features[image_name] = np.stack(feature_list)
    return features
features = extract_features(points_dic)
print(features)

d:\Programs\anaconda3\envs\qsynch1\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Programs\anaconda3\envs\qsynch1\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


{'060_0000': array([[0.25872564, 0.16466251, 0.1666457 , ..., 1.7793053 , 0.3410358 ,
        0.20368606],
       [0.05972493, 0.1827132 , 0.19878282, ..., 0.28849053, 0.40006065,
        0.19222704],
       [0.07381159, 0.08972877, 0.24928649, ..., 0.5657842 , 0.22115655,
        0.03051845],
       [0.53599703, 0.14482543, 0.05216942, ..., 0.15096366, 0.30161628,
        0.07334161]], dtype=float32), '060_0010': array([[0.56656635, 0.17640263, 0.21756753, ..., 0.40727705, 0.59511966,
        0.22087309],
       [0.9113531 , 0.04289479, 0.38794434, ..., 0.23997077, 0.54686576,
        0.34603283],
       [0.87234306, 0.01721076, 0.4082014 , ..., 0.38801634, 0.3431101 ,
        0.10698917],
       [0.8535742 , 0.08141699, 0.26365814, ..., 0.02243344, 0.5071528 ,
        0.09732227]], dtype=float32), '060_0013': array([[0.5342985 , 0.13998728, 0.3579746 , ..., 0.2181499 , 1.7884122 ,
        0.2735663 ],
       [0.7123156 , 0.24274682, 0.38634437, ..., 0.02257181, 1.757765  ,
        0.

In [5]:
#features have been extracted, now we need to calculate the distance matrix
def pairwise_permutations(features_dict, pm_size: int) -> dict: # compute the P_ij
    P = {}
    image_list = list(features_dict.keys())

    for i in range(len(image_list)):
        key0= f"P{i+1}{i+1}"
        P[key0] = np.eye(pm_size)
        for j in range(i + 1, len(image_list)):
            img1 = image_list[i]
            img2 = image_list[j]

            feats1 = features_dict[img1]
            feats2 = features_dict[img2]
            similarity = cosine_similarity(feats1, feats2) # equivalent to cosine similarity, since the features are normalized
            print("this is the similarity matrix")
            print(similarity)
            cost_mat = (1 - similarity)
            row_ind, col_ind = linear_sum_assignment(cost_mat)
            n = len(row_ind)
            perm_matrix = np.zeros((n, n), dtype=int)
            perm_matrix[row_ind, col_ind] = 1
            key1 = f"P{i+1}{j+1}"
            P[key1] = perm_matrix
            key2 = f"P{j+1}{i+1}"
            P[key2] = perm_matrix.T
    return P
P = pairwise_permutations(features, num_keypoints)
print(len(P))
print(P)

this is the similarity matrix
[[0.6701941  0.642549   0.68861943 0.5146345 ]
 [0.64447176 0.6879388  0.7465936  0.63020587]
 [0.6588704  0.7086817  0.7669008  0.63553977]
 [0.72410357 0.757544   0.7735055  0.83913565]]
this is the similarity matrix
[[0.5812422  0.6007061  0.5711621  0.5714956 ]
 [0.61445177 0.69406843 0.65799785 0.64671457]
 [0.5985336  0.6793962  0.6344446  0.64115703]
 [0.69602925 0.7377752  0.7075008  0.85892713]]
this is the similarity matrix
[[0.8063059  0.82841873 0.7647438  0.7298002 ]
 [0.7953354  0.8799246  0.7993521  0.7950195 ]
 [0.7868025  0.85370827 0.789339   0.7956416 ]
 [0.69648725 0.7390039  0.7241779  0.87714434]]
9
{'P11': array([[1., 0., 0., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.]]), 'P12': array([[1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]]), 'P21': array([[1, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [0, 0, 0, 1]]), 'P13': array([[1, 0, 0, 0],
       [0, 0, 1, 

In [6]:
def qubo_form_maker_fixed_gauge(P: dict,
                                num_views: int,
                                num_keypoints: int,
                                penalty: float = 2.5) -> tuple[np.ndarray, np.ndarray]:
    """
    Constructs the QUBO matrix Q and linear vector s for permutation synchronization,
    fixing the gauge by setting the first absolute permutation matrix X1 = I.

    Args:
        P: Dictionary of pairwise permutations Pij, e.g., P['P12'], P['P21'].
           Keys are 1-based indexed, e.g., "P12", "P23".
        num_views: Total number of views (m).
        num_keypoints: Number of keypoints per view (n).
        penalty: Lambda value for the constraint penalty term.

    Returns:
        A tuple containing the QUBO matrix Q and linear vector s for the
        optimization variables corresponding to X2, ..., Xm.
    """
    m = num_views
    n = num_keypoints
    N = n * n # Number of variables per view matrix (n^2)

    if m <= 1:
        # Nothing to optimize if there's only one view (or zero)
        return np.array([[]]), np.array([])

    m_opt = m - 1 # Number of views to optimize (X2 to Xm)
    num_vars = m_opt * N # Total number of binary variables in the QUBO

    # --- Initialize QUBO components for optimization variables ---
    Q_prime_opt = np.zeros((num_vars, num_vars))
    s_opt_initial = np.zeros(num_vars) # Linear terms arising from fixing X1=I

    # --- Construct the constant vector for vec(X1) = vec(I) ---
    x1 = np.zeros(N)
    for k in range(n):
        x1[k * n + k] = 1

    # --- Populate Q_prime_opt and s_opt_initial from the objective function ---
    # Objective: sum_{i,j} ||Pij - Xi Xj^T||^2 = const - 2 * sum_{i,j} tr(Xj X_i^T Pij)
    # We want to minimize: - sum_{i,j} vec(Xi)^T (I kron Pij) vec(Xj)

    for key, p_mat_ij in P.items():
        # Extract i and j (0-based index for internal calculations)
        # Assumes keys are like 'P12', 'P21', etc. (1-based)
        try:
            i = int(key[1]) - 1
            j = int(key[2]) - 1
        except (IndexError, ValueError):
            print(f"Warning: Could not parse view indices from key '{key}'. Skipping.")
            continue

        # Calculate the block matrix for the interaction term
        block_ij = np.kron(np.eye(n), p_mat_ij) # This is (I kron Pij)

        if i > 0 and j > 0:
            # Interaction between Xi and Xj (where i, j != 1)
            # Adjust indices for the optimization variables (X2...Xm map to indices 0...m_opt-1)
            opt_i_idx = i - 1
            opt_j_idx = j - 1
            start_row = opt_i_idx * N
            end_row = start_row + N
            start_col = opt_j_idx * N
            end_col = start_col + N
            # Subtract term based on -vec(Xi)^T * block_ij * vec(Xj)
            Q_prime_opt[start_row:end_row, start_col:end_col] -= block_ij

        elif i == 0 and j > 0:
            # Interaction between X1 (Identity) and Xj
            # Term is -vec(X1)^T * block_1j * vec(Xj) -> contributes to linear part of Xj
            opt_j_idx = j - 1
            linear_contrib = -x1.T @ block_ij # This is a row vector, needs reshaping? No, result is scalar * vec(Xj) coefs.
            start_col = opt_j_idx * N
            end_col = start_col + N
            s_opt_initial[start_col:end_col] += linear_contrib

        elif i > 0 and j == 0:
            # Interaction between Xi and X1 (Identity)
            # Term is -vec(Xi)^T * block_i1 * vec(X1) -> contributes to linear part of Xi
            opt_i_idx = i - 1
            # Need vec(Xi)^T @ block_i1 @ x1 = (block_i1 @ x1)^T @ vec(Xi)
            linear_contrib = -(block_ij @ x1) # block_ij here is (I kron Pi1)
            start_row = opt_i_idx * N
            end_row = start_row + N
            s_opt_initial[start_row:end_row] += linear_contrib

        # Case i=0 and j=0 corresponds to tr(X1 X1^T P11) = tr(I I I) = n, a constant, ignore.

    # --- Construct the constraint matrix A_opt and vector b_opt ---
    # Constraints only apply to X2, ..., Xm
    num_constraints = m_opt * 2 * n # Each of m_opt matrices has n row + n col constraints
    A_opt = np.zeros((num_constraints, num_vars))
    b_opt = np.ones(num_constraints) # All constraints require sums to be 1

    current_constraint_row = 0
    for view_idx_opt in range(m_opt): # Iterate through X2...Xm (indexed 0 to m_opt-1)
        view_offset = view_idx_opt * N

        # Row sum constraints for current view
        for row in range(n):
            for col in range(n):
                var_index = view_offset + row * n + col
                A_opt[current_constraint_row, var_index] = 1
            current_constraint_row += 1

        # Column sum constraints for current view
        for col in range(n):
            for row in range(n):
                var_index = view_offset + row * n + col
                A_opt[current_constraint_row, var_index] = 1
            current_constraint_row += 1

    # --- Calculate final Q and s ---
    Q = Q_prime_opt + penalty * (A_opt.T @ A_opt)
    s = s_opt_initial - 2 * penalty * (A_opt.T @ b_opt)

    return Q, s

In [7]:
Q,s = qubo_form_maker_fixed_gauge(P,num_views,num_keypoints)
print(Q)
print(s)
print(len(s))

[[4.  2.5 2.5 ... 0.  0.  0. ]
 [2.5 4.  2.5 ... 0.  0.  0. ]
 [2.5 2.5 4.  ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 4.  2.5 2.5]
 [0.  0.  0.  ... 2.5 4.  2.5]
 [0.  0.  0.  ... 2.5 2.5 4. ]]
[-12. -10. -10. -10. -10. -10. -12. -10. -10. -12. -10. -10. -10. -10.
 -10. -12. -12. -10. -10. -10. -10. -10. -12. -10. -10. -12. -10. -10.
 -10. -10. -10. -12.]
32


In [ ]:
# import scipy.sparse as sp

# if not hasattr(sp.csr_matrix, "_mul_vector") and hasattr(sp.csr_matrix, "_matmul_vector"):
#     # expose the new name under the old one so quimb can find it
#     sp.csr_matrix._mul_vector = sp.csr_matrix._matmul_vector
#     sp.csc_matrix._mul_vector = sp.csc_matrix._matmul_vector
#     sp.coo_matrix._mul_vector = sp.coo_matrix._matmul_vector

In [8]:
from __future__ import annotations

import scipy.optimize as opt
from blueqat import Circuit
from blueqat.pauli import qubo_bit as q

In [ ]:
__all__ = [
    "build_blueqat_hamiltonian",
    "make_expectation",
    "run_qaoa",
]

In [9]:
#Using blueQat instead
def _to_float(x):
    """Return *scalar* float even if x is a 0‑D/1‑D NumPy number."""
    return float(np.asarray(x))


def build_blueqat_hamiltonian(Q: np.ndarray, s: np.ndarray):
    """Convert ``cost = xᵀ Q x + sᵀ x`` to a Blueqat PauliSum.

    * Diagonal of *Q* is absorbed into the linear part because *x_i² = x_i*.
    * Off‑diagonal is taken from upper‑triangle only.
    * All coefficients are cast to plain Python ``float`` so that Blueqat’s
      PauliTerm constructor never receives a NumPy scalar (this was the source
      of the previous ValueError).
    """
    Q = np.asarray(Q, dtype=float)
    s = np.asarray(s, dtype=float).ravel()
    n = Q.shape[0]

    H = 0  # PauliSum accumulator ------------------------------------------------

    # linear (includes the diagonal of Q)
    for i in range(n):
        coeff = _to_float(s[i] + Q[i, i])
        if coeff:
            H += coeff * q(i)

    # quadratic (strictly upper‑triangle)
    for i in range(n):
        for j in range(i + 1, n):
            coeff = _to_float(Q[i, j])
            if coeff:
                H += coeff * q(i) * q(j)

    return H.simplify()

In [ ]:
# H = build_blueqat_hamiltonian(Q,s)
# print(H)

In [10]:
# -----------------------------------------------------------------------------
# 2.  Expectation builder
# -----------------------------------------------------------------------------

def make_expectation(H, time_evolutions, p: int, backend="quimb"):
    """Return ``f(params)`` that computes ⟨ψ(β⃗,γ⃗)|H|ψ(β⃗,γ⃗)⟩."""
    n_qubits = H.n_qubits

    def f(params):
        beta = params[:p]
        gamma = params[p:]
        print(f"Evaluating with beta={beta}, gamma={gamma}")

        c = Circuit(n_qubits).h[:]
        for layer in range(p):
            for evo in time_evolutions:
                evo(c, gamma[layer])
            c.rx(beta[layer])[:]
        
        result = c.run(backend=backend, hamiltonian=H)
        print(f"Expectation value: {result}")
        return result

    return f

In [11]:
def run_qaoa(
    Q: np.ndarray,
    s: np.ndarray,
    p: int = 1,
    method: str = "L-BFGS-B",
    tol: float | None = 1e-3,
    random_seed: int | None = None,
    backend: str = "quimb",
):
    """Optimise p‑layer QAOA.

    Returns ``(best_expectation, scipy_result)``.
    """
    print("Starting QAOA optimization...")
    
    if random_seed is not None:
        print("Setting random seed...")
        np.random.seed(random_seed)

    print("Building Hamiltonian...")
    H = build_blueqat_hamiltonian(Q, s)

    print("Calculating time evolutions...")
    time_evos = [t.get_time_evolution() for t in H]

    print("Preparing expectation function...")
    f = make_expectation(H, time_evos, p, backend=backend)

    print("Initializing angles and starting optimization...")
    init = 2 * np.pi * np.random.rand(2 * p)
    res = opt.minimize(f, init, method=method, tol=tol)

    if not res.success:
        raise RuntimeError(res.message)

    print("Optimization complete.")
    return res.fun, res


In [12]:
Eopt, result = run_qaoa(Q, s, p=1, random_seed=42)
print("Optimised ⟨H⟩:", Eopt)
print("β, γ:", result.x)

Starting QAOA optimization...
Setting random seed...
Building Hamiltonian...
Calculating time evolutions...
Preparing expectation function...
Initializing angles and starting optimization...
Evaluating with beta=[2.35330497], gamma=[5.97351416]
Expectation value: -48.39008446747264
Evaluating with beta=[2.35330498], gamma=[5.97351416]
Expectation value: -48.390084467796186
Evaluating with beta=[2.35330497], gamma=[5.97351417]
Expectation value: -48.39008441219596
Evaluating with beta=[2.35915807], gamma=[4.97353129]
Expectation value: -56.90139252866536
Evaluating with beta=[2.35915808], gamma=[4.97353129]
Expectation value: -56.90139240687967
Evaluating with beta=[2.35915807], gamma=[4.9735313]
Expectation value: -56.901392685266096
Evaluating with beta=[2.35482691], gamma=[5.7134969]
Expectation value: -52.0104923782405
Evaluating with beta=[2.35482692], gamma=[5.7134969]
Expectation value: -52.010492388778644
Evaluating with beta=[2.35482691], gamma=[5.71349691]
Expectation value: -

In [13]:
print(Eopt)

-63.98124320206259


In [14]:
print(result)

  message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
  success: True
   status: 0
      fun: -63.98124320206259
        x: [ 1.498e+00  5.027e+00]
      nit: 5
      jac: [ 1.820e-02  4.387e+00]
     nfev: 33
     njev: 11
 hess_inv: <2x2 LbfgsInvHessProduct with dtype=float64>


In [16]:
# H = build_blueqat_hamiltonian(Q, s)

# 2.  Get the list of callables that apply e^{-i γ H_j} for each term
time_evos = [term.get_time_evolution() for term in H]
print(time_evos)

[<function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FE4D0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FE8C0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FE950>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FE9E0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FEA70>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FEB00>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FEB90>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FEC20>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FECB0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FED40>, <function Term.get_time_evolution.<locals>.append_to_circuit at 0x000001BE563FEDD0>, <function Term.get_time_evolution.<locals>.append_to_circuit at 

In [21]:
from blueqat import Circuit

n = Q.shape[0]                       # number of binary variables

def make_circuit(betas, gammas):
    c = Circuit(n).h[:]              # start in |+⟩⊗ⁿ
    for β, γ in zip(betas, gammas):
        for evo in time_evos:        # the time_evos list you built earlier
            evo(c, γ)                # cost unitary
        c.rx(β)[:]                  # mixer
    return c.m[:]

# best angles from the optimiser
betas  = [1.49849317]
gammas = [5.02681834]

shots = 10
counts = make_circuit(betas, gammas).run(shots=shots, backend="quimb")


In [22]:
print(counts)

Counter({'10000010010000001000001001000001': 1, '00000010010000010000001001000001': 1, '10100010011100011010001001110001': 1, '11000010110100011100001111010001': 1, '10011010010000011001101001000001': 1, '10000010010000011000001001000001': 1, '10000110010100111000011001011011': 1, '00000010010000010000001101000001': 1, '10000010010100011000001001010001': 1, '00010010010010000001001011001000': 1})


In [ ]:
# --- decode & score ---------------------------------------------------
def decode(bits):
    x = np.fromiter(map(int, bits), dtype=int)
    return bitvector_to_perms(x, num_views, num_keypoints)   # your helper

best = max(counts, key=counts.get)         # most frequent bit‑string
solution = decode(best)
print("Most frequent string:", best)
print("Decoded permutation set shape:", solution.shape)
print("Original objective value:", soln_cost(solution))      # your scorer

# From here on out is the old version

In [ ]:
# Necessary imports
import numpy as np
from qiskit_optimization import QuadraticProgram
from qiskit.quantum_info import SparsePauliOp
#

def construct_qaoa_hamiltonian_from_qubo(Q: np.ndarray, s: np.ndarray) -> tuple[SparsePauliOp, float]:
    """
    Constructs the Ising Hamiltonian (Problem Hamiltonian for QAOA)
    from a QUBO formulation (Q matrix and s vector).

    Args:
        Q: The square matrix representing quadratic terms in the QUBO.
           Assumes Q is defined for the variables being optimized (e.g., X2...Xm
           if X1 is fixed).
        s: The vector representing linear terms in the QUBO for the same variables.

    Returns:
        A tuple containing:
        - op (SparsePauliOp): The Qiskit SparsePauliOp representing the Ising Hamiltonian.
        - offset (float): The constant offset term resulting from the QUBO-to-Ising conversion.
                          This shifts the energy levels but doesn't affect the optimal solution.
    """
    num_vars = Q.shape[0]
    if num_vars == 0:
        # Handle empty case
        return SparsePauliOp(['I' * 0], coeffs=[0]), 0.0
    if Q.shape != (num_vars, num_vars) or s.shape != (num_vars,):
        raise ValueError("Dimensions of Q and s do not match.")

    # 1. Create a QuadraticProgram object
    qp = QuadraticProgram("permutation_sync_qubo")

    # 2. Add binary variables corresponding to the elements of the vectorized Xi's
    #    The number of variables is the dimension of s (or Q)
    for i in range(num_vars):
        qp.binary_var(f'x{i}')

    # 3. Define the objective function from Q and s
    #    QuadraticProgram expects a dictionary format for linear and quadratic terms.
    linear_coeffs = {f'x{i}': s[i] for i in range(num_vars)}

    # Qiskit's QuadraticProgram uses the upper triangle of Q by default.
    # Ensure Q is symmetric or provide the full Q. Let's build the dict carefully.
    quadratic_coeffs = {}
    for i in range(num_vars):
        for j in range(i, num_vars): # Iterate through upper triangle including diagonal
            if not np.isclose(Q[i, j], 0.0):
                 # For Qiskit QP: diagonal terms Q[i,i] go here directly
                 # Off-diagonal Q[i,j] (i<j) go here directly
                 # Qiskit handles the conversion factor internally in to_ising()
                 quadratic_coeffs[(f'x{i}', f'x{j}')] = Q[i, j]


    qp.minimize(linear=linear_coeffs, quadratic=quadratic_coeffs)

    print("Quadratic Program defined:")
    print(qp.export_as_lp_string()) # Optional: print LP format for verification

    # 4. Convert the QuadraticProgram to an Ising Hamiltonian
    #    The to_ising() method directly performs the transformation:
    #    x_i = (Z_i + 1) / 2
    #    It returns the operator and the energy offset.
    op, offset = qp.to_ising()

    # Ensure the operator is SparsePauliOp (standard in modern Qiskit)
    # If using older Qiskit, op might be PauliSumOp, conversion might be needed
    # if isinstance(op, PauliSumOp):
    #     op = op.to_spmatrix() # Or convert as needed for your QAOA implementation

    print(f"\nIsing Hamiltonian Operator (SparsePauliOp):\n{op}")
    print(f"\nEnergy Offset: {offset}")

    return op, offset



In [ ]:
# dummy hamiltonian test
# --- Example Usage ---

# Create dummy Q and s (replace with output from your qubo_form_maker_fixed_gauge)
# Example: 2 views (X1 fixed, optimize X2), 2 keypoints (n=2)
# num_vars = n*n * (m-1) = 2*2 * (2-1) = 4
dummy_Q = np.array([
    [ 1, -1,  0,  0.5],
    [-1,  1,  0.5, 0],
    [ 0,  0.5, 1, -1],
    [ 0.5, 0, -1,  1]
])
dummy_s = np.array([-0.5, 0.5, -0.5, 0.5])

print("--- Constructing Hamiltonian ---")
hamiltonian_op, energy_offset = construct_qaoa_hamiltonian_from_qubo(dummy_Q, dummy_s)
print(hamiltonian_op)
print(energy_offset)
# Now you can use 'hamiltonian_op' as the problem Hamiltonian (H_P)
# in your QAOA implementation (e.g., with Qiskit's QAOA class or VQE).
# Remember QAOA also requires a mixer Hamiltonian (H_M), typically sum of Pauli X's.

In [ ]:
#real one
print("--- Constructing Hamiltonian ---")
hamiltonian_op, energy_offset = construct_qaoa_hamiltonian_from_qubo(Q, s)
print(hamiltonian_op)
print(energy_offset)

In [ ]:
def extract_pauli_terms(op: SparsePauliOp):
    """
    Extracts the list of (coefficient, pauli_string) from Qiskit's SparsePauliOp
    for use in external simulators like quimb.
    
    Args:
        op (SparsePauliOp): The Ising Hamiltonian as SparsePauliOp.

    Returns:
        List of tuples (coeff, pauli_string) where pauli_string is a string like 'ZIZI'
    """
    terms = []
    for p, c in zip(op.paulis, op.coeffs):
        terms.append((float(np.real(c)), p.to_label()))
    return terms

# run usage
terms = extract_pauli_terms(hamiltonian_op)
print(terms)

In [ ]:
#using quimb


import numpy as np
import quimb as qu
import quimb.tensor as qtn


def apply_cost_unitary(circuit, terms, gamma):
    """
    Applies the cost unitary exp(-i * gamma * H_C) using the Pauli terms.

    Args:
        circuit: A quimb QuantumCircuit object
        terms: List of (coeff, Pauli string)
        gamma: QAOA cost parameter
    """
    for coeff, pauli in terms:
        angle = gamma * coeff  # The angle for the full exponential i.e., exp(-i * angle * Pauli)
        qubits = [i for i, p in enumerate(pauli) if p != 'I']
        pauli_ops = [p for p in pauli if p != 'I']

        if len(qubits) == 1:
            # Apply exp(-i * angle * Pauli) for single-qubit term
            pauli_type = pauli_ops[0] # 'X', 'Y', or 'Z'
            if pauli_type == 'Z':
                # RZ(theta) applies exp(-i * theta * Z / 2), need theta/2 = angle
                circuit.apply_gate("RZ", 2 * angle, qubits[0])
            elif pauli_type == 'X':
                # RX(theta) applies exp(-i * theta * X / 2), need theta/2 = angle
                circuit.apply_gate("RX", 2 * angle, qubits[0])
            elif pauli_type == 'Y':
                 # RY(theta) applies exp(-i * theta * Y / 2), need theta/2 = angle
                circuit.apply_gate("RY", 2 * angle, qubits[0])
            else:
                 raise NotImplementedError(f"Single-qubit Pauli term {pauli_type} not supported.")

        elif len(qubits) == 2:
            # Example: for 'ZZ', apply ZZ rotation
            if ''.join(pauli_ops) == 'ZZ':
                # RZZ(theta) applies exp(-i * theta * ZZ / 2), need theta/2 = angle
                circuit.apply_gate("RZZ", 2 * angle, *qubits)
            else:
                # Can be extended for XX, YY, etc. if needed
                # These would typically be decomposed into CNOTs and single-qubit rotations
                raise NotImplementedError(f"Two-qubit Pauli term {pauli} not supported in direct gate.")

        elif len(qubits) == 0:
            pass  # Identity term, just a global phase (often ignored in optimization)
        else:
            raise NotImplementedError(f"Multi-qubit term {pauli} not supported.")


def apply_mixer_unitary(circuit, beta, n_qubits):
    """
    Applies the standard mixer Hamiltonian for QAOA: exp(-i * beta * X_i)
    """
    # Applying exp(-i * beta * X_i) to each qubit i
    # RX(theta) applies exp(-i * theta * X / 2)
    # We need theta / 2 = beta, so theta = 2 * beta
    for i in range(n_qubits):
        circuit.apply_gate("RX", 2 * beta, i)


def build_qaoa_circuit(n_qubits, terms, gamma, beta):
    """
    Build a QAOA circuit with 1 layer.
    
    Args:
        n_qubits: Number of qubits
        terms: Pauli terms (from extract_pauli_terms)
        gamma: Cost parameter
        beta: Mixer parameter
    
    Returns:
        A TensorNetwork circuit object
    """
    circuit = qtn.Circuit(n_qubits)
    
    # Step 1: Start in uniform superposition
    for i in range(n_qubits):
        circuit.apply_gate("H", i)
    
    # Step 2: Apply cost Hamiltonian
    apply_cost_unitary(circuit, terms, gamma)
    
    # Step 3: Apply mixer Hamiltonian
    apply_mixer_unitary(circuit, beta, n_qubits)
    
    return circuit
# run usage
n_qubits = len(terms[0][1])  # infer from Pauli string
gamma, beta = 0.5, 0.5

circuit = build_qaoa_circuit(n_qubits, terms, gamma, beta)

In [ ]:
print(circuit)